# Parameteriser

Brenda API (requires user to create login)  
https://www.brenda-enzymes.org/soap.php

In [ ]:
# %pip install zeep
# %pip install ptitprince

In [ ]:
import hashlib
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde
from sspipe import p
from zeep import Client


@dataclass
class Km:
    value: float
    substrate: str
    organism: str
    commentary: str | None
    literature: list[str]


def normalise(x: np.ndarray) -> np.ndarray:
    return x / np.sum(x)


def data_coordinates_of_width(ax, width):
    return ax.transData.inverted().transform(ax.transAxes.transform([width, 0]))[0]


def data_coordinates_of_height(ax, height):
    return ax.transData.inverted().transform(ax.transAxes.transform([0, height]))[1]


def data_to_display(ax, val):
    return ax.transData.transform(val)


def display_to_data(ax, val):
    return ax.transData.inverted().transform(val)


def axes_to_display(ax, val):
    return ax.transAxes.transform(val)


def display_to_axes(ax, val):
    return ax.transAxes.inverted().transform(val)


def x_to_data(ax, val):
    return ax.transLimits.inverted().transform((val, 0))[0]


def y_to_data(ax, val):
    return ax.transLimits.inverted().transform((0, val))[1]


def add_boxplot(ax, data, color="C0", offset: int = 0):
    _d = data.describe()
    iqr = _d["75%"] - _d["25%"]
    height = y_to_data(ax, 0.05)

    # Box
    ax.add_artist(
        Rectangle(
            (_d["25%"], offset * height),
            width=_d["75%"] - _d["25%"],
            height=height,
            facecolor=color,
            linewidth=1.5,
            alpha=0.7,
        )
    )

    # Bars
    ax.add_artist(
        Line2D(
            xdata=[data.median()],
            ydata=[offset * height, offset * height + height],
            color="white",
        )
    )

    # Whiskers
    ax.add_artist(
        Line2D(
            xdata=[max(_d["25%"] - 1.5 * iqr, ax.get_xlim()[0]), _d["25%"]],
            ydata=[offset * height + height / 2],
            color=color,
            alpha=0.7,
        )
    )
    ax.add_artist(
        Line2D(
            xdata=[_d["75%"], _d["75%"] + 1.5 * iqr],
            ydata=[offset * height + height / 2],
            color=color,
            alpha=0.7,
        )
    )


with open(".env") as fp:
    cred: dict[str, str] = dict(
        line.split("=", maxsplit=1) for line in fp.read().strip().split("\n")
    )

In [ ]:
@dataclass
class Brenda:
    email: str
    password: str
    tmp_dir: Path = Path(".") / "tmp"
    wsdl: str = "https://www.brenda-enzymes.org/soap/brenda_zeep.wsdl"

    def __post_init__(self) -> None:
        self.password = hashlib.sha256(self.password.encode("utf-8")).hexdigest()
        self.tmp_dir.mkdir(exist_ok=True, parents=True)

    def get_km(self, ec_number: str, verbose: bool = False) -> pd.DataFrame:
        # organism: str | None = None
        # if organism is None:
        organism = ""

        filename = self.tmp_dir / f"km-{ec_number}.json"

        if filename.exists():
            if verbose:
                print("Using cached data")
            with open(filename, "r", encoding="utf-8") as fp:
                data = [Km(**i) for i in json.load(fp)]
        else:
            if verbose:
                print("Downloading data ...")
            data = [
                Km(
                    value=float(res["kmValue"]),
                    substrate=res["substrate"],
                    organism=res["organism"],
                    commentary=res["commentary"],
                    literature=res["literature"],
                )
                for res in Client(self.wsdl).service.getKmValue(
                    self.email,
                    self.password,
                    f"ecNumber*{ec_number}",
                    f"organism*{organism}",
                    "kmValue*",
                    "kmValueMaximum*",
                    "substrate*",
                    "commentary*",
                    "ligandStructureId*",
                    "literature*",
                )
            ]
            with open(filename, "w+", encoding="utf-8") as fp:
                json.dump([asdict(i) for i in data], fp)

        return pd.DataFrame(data)


brenda = Brenda(email=cred["EMAIL"], password=cred["PASSWORD"])
(kms := brenda.get_km("4.1.1.39", verbose=True)).head()

In [ ]:
by_substrate = {
    key: df.drop(columns=["substrate"]) for key, df in kms.groupby("substrate")
}

df = by_substrate["CO2"]
df.head()

In [ ]:
s = df["value"]

x = np.linspace(s.min(), s.max(), 1001)
y = gaussian_kde(s)(x) | p(normalise)

fig, ax = plt.subplots()
ax.fill_between(x, y, alpha=0.2)
ax.plot(x, y)
ax.set_xscale("log")

In [ ]:
filtered = s[s < np.percentile(s, 95)]
x = np.geomspace(filtered.min(), filtered.max(), 1001)
y = gaussian_kde(filtered)(x) | p(normalise)

fig, ax = plt.subplots()
ax.set_title("Km - 4.1.1.39 - CO2")
ax.fill_between(x, y, alpha=0.2)
ax.plot(x, y)
ax.set_xscale("log")

In [ ]:
x = np.geomspace(filtered.min(), filtered.max(), 1001)
organism = df[df["organism"] == "Nicotiana tabacum"]["value"]

y1 = gaussian_kde(filtered)(x) | p(normalise)
y2 = gaussian_kde(organism)(x) | p(normalise)

with plt.rc_context(
    {
        "grid.color": "0.8",
        "xtick.color": "0.8",
        "ytick.color": "0.8",
        "xtick.labelcolor": "0.3",
        "ytick.labelcolor": "0.3",
    }
):
    fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
    ax.set_title("Km - 4.1.1.39 - CO2")
    ax.set_xlim(filtered.min(), filtered.max())
    ax.set_ylim(0, max(y1.max(), y2.max()) * 1.1)
    ax.set_xscale("log")

    ax.fill_between(x, y1, alpha=0.2)
    ax.fill_between(x, y2, alpha=0.2)
    ax.plot(x, y1, label="All")
    ax.plot(x, y2, label="Nicotiana tabacum")
    ax.legend()
    add_boxplot(ax, filtered, "C0")
    add_boxplot(ax, organism, "C1", offset=1)
    ax.grid()
    ax.set_frame_on(False)